# 04 - Date Range Exploration

This notebook analyzes the time coverage of the sales and customer data.

Focus areas:
- First and latest order dates
- Historical range of sales data
- Customer age range
- Sales activity by year and month
- Delivery and shipping time patterns
- Order recency analysis

In [0]:
%sql
/*
Sales Date Range
----------------
Purpose:
    Identify the first and latest order dates in the sales fact table.
    This shows the historical coverage of the sales dataset.
*/

SELECT 
    MIN(order_date) AS first_order_date,
    MAX(order_date) AS last_order_date,
    ROUND(MONTHS_BETWEEN(MAX(order_date), MIN(order_date)), 0) AS order_range_months,
    DATEDIFF(MAX(order_date), MIN(order_date)) AS order_range_days
FROM datawarehouseanalytics_gold.fact_sales;

In [0]:
%sql
/*
Customer Age Range
Purpose:
    Identify the oldest and youngest customers based on birthdate.
    Customer age is calculated using the current date.
*/

SELECT
    MIN(birthdate) AS oldest_birthdate,
    FLOOR(MONTHS_BETWEEN(CURRENT_DATE(), MIN(birthdate)) / 12) AS oldest_age,
    MAX(birthdate) AS youngest_birthdate,
    FLOOR(MONTHS_BETWEEN(CURRENT_DATE(), MAX(birthdate)) / 12) AS youngest_age
FROM datawarehouseanalytics_gold.dim_customers;

In [0]:
%sql
/*
Yearly Sales Activity

Purpose:
    Analyze sales activity by year using records with valid order dates.
    Null order dates are excluded because they cannot be assigned to a sales year.
*/

SELECT
    YEAR(order_date) AS order_year,
    COUNT(DISTINCT order_number) AS total_orders,
    SUM(quantity) AS total_quantity,
    SUM(sales_amount) AS total_sales
FROM datawarehouseanalytics_gold.fact_sales
WHERE order_date IS NOT NULL
GROUP BY YEAR(order_date)
ORDER BY order_year;

In [0]:
%sql
/*
Monthly Sales Trend
Purpose:
    Analyze monthly revenue trends across the full sales period.
    DATE_TRUNC is used to group orders at month level.
*/

SELECT
    DATE_TRUNC('MONTH', order_date) AS order_month,
    COUNT(DISTINCT order_number) AS total_orders,
    SUM(sales_amount) AS total_sales,
    SUM(quantity) AS total_quantity
FROM datawarehouseanalytics_gold.fact_sales
WHERE order_date IS NOT NULL
GROUP BY DATE_TRUNC('MONTH', order_date)
ORDER BY order_month;

In [0]:
%sql
/*
Shipping Duration Analysis
Purpose:
    Measure the time taken between order date and shipping date.
    This helps understand basic fulfillment patterns.
*/

SELECT
    MIN(DATEDIFF(shipping_date, order_date)) AS min_shipping_days,
    MAX(DATEDIFF(shipping_date, order_date)) AS max_shipping_days,
    ROUND(AVG(DATEDIFF(shipping_date, order_date)), 2) AS avg_shipping_days
FROM datawarehouseanalytics_gold.fact_sales;

In [0]:
%sql
/*
Shipping Duration Analysis

Purpose:
    Measure the time taken between order date and shipping date.
    This helps understand basic fulfillment patterns.
*/

SELECT
    MIN(DATEDIFF(shipping_date, order_date)) AS min_shipping_days,
    MAX(DATEDIFF(shipping_date, order_date)) AS max_shipping_days,
    ROUND(AVG(DATEDIFF(shipping_date, order_date)), 2) AS avg_shipping_days
FROM datawarehouseanalytics_gold.fact_sales;

In [0]:
%sql
/*
Due Date Delay Check

Purpose:
    Check whether any orders were shipped after the due date.
    This helps identify possible late fulfillment records.

*/

SELECT
    COUNT(*) AS total_sales_lines,
    SUM(
        CASE 
            WHEN shipping_date > due_date THEN 1 
            ELSE 0 
        END
    ) AS delayed_sales_lines,
    ROUND(
        SUM(CASE WHEN shipping_date > due_date THEN 1 ELSE 0 END) * 100.0 / COUNT(*),
        2
    ) AS delayed_percentage
FROM datawarehouseanalytics_gold.fact_sales;